<a href="https://colab.research.google.com/github/Muralikrishna97/-Fitness-Planner/blob/main/Russia_Ukraine_War_Reddit_Sentiment_5_Model_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# RUSSIA-UKRAINE WAR REDDIT SENTIMENT ANALYSIS
# 5-MODEL COMPARISON
# ============================================================

print("=" * 80)
print("RUSSIA-UKRAINE WAR REDDIT SENTIMENT ANALYSIS")
print("5-MODEL COMPARISON")
print("=" * 80)

print("""
Objective:
Compare five machine learning and deep learning approaches
for sentiment classification of Reddit comments related to
the Russia-Ukraine War.

Models:
1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM
3. CNN
4. BERT
5. BERT-CNN + Topic

Dataset:
Reddit comments from the dataset used in the referenced study.

Sentiment labels:
VADER-generated Positive, Neutral, and Negative labels.

Evaluation:
All models will use the same frozen train, validation,
and test partitions for a fair comparison.
""")

RUSSIA-UKRAINE WAR REDDIT SENTIMENT ANALYSIS
5-MODEL COMPARISON

Objective:
Compare five machine learning and deep learning approaches
for sentiment classification of Reddit comments related to
the Russia-Ukraine War.

Models:
1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM
3. CNN
4. BERT
5. BERT-CNN + Topic

Dataset:
Reddit comments from the dataset used in the referenced study.

Sentiment labels:
VADER-generated Positive, Neutral, and Negative labels.

Evaluation:
All models will use the same frozen train, validation,
and test partitions for a fair comparison.



In [7]:
# ============================================================
# LOAD FROZEN TRAIN / VALIDATION / TEST DATA
# ============================================================

import pandas as pd

TRAIN_PATH = "/content/reddit_train.csv"
VAL_PATH   = "/content/reddit_validation.csv"
TEST_PATH  = "/content/reddit_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("=" * 70)
print("DATASETS LOADED")
print("=" * 70)

print(f"Train:       {len(train_df):,} comments")
print(f"Validation:  {len(val_df):,} comments")
print(f"Test:        {len(test_df):,} comments")
print(f"Total:       {len(train_df) + len(val_df) + len(test_df):,} comments")

print("\nColumns:")
print(train_df.columns.tolist())

print("\nFirst 3 training comments:")
display(train_df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: '/content/reddit_train.csv'

In [ ]:
# ============================================================
# DATASET AND LABEL VERIFICATION
# ============================================================

import pandas as pd

LABELS = ["Negative", "Neutral", "Positive"]

print("=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Missing values
# ------------------------------------------------------------

print("\nMissing values:")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(df[["comment", "sentiment"]].isna().sum())

# ------------------------------------------------------------
# 2. Sentiment distributions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SENTIMENT DISTRIBUTION")
print("=" * 70)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")

    counts = df["sentiment"].value_counts()
    percentages = df["sentiment"].value_counts(normalize=True) * 100

    distribution = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    print(distribution)

# ------------------------------------------------------------
# 3. Check for unexpected labels
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LABEL CHECK")
print("=" * 70)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    unexpected = set(df["sentiment"].dropna().unique()) - set(LABELS)

    if unexpected:
        print(f"⚠ {name}: unexpected labels → {unexpected}")
    else:
        print(f"✓ {name}: only expected labels found")

# ------------------------------------------------------------
# 4. Check empty comments
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EMPTY COMMENT CHECK")
print("=" * 70)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    empty = (
        df["comment"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{name}: {empty} empty comments")

print("\n✓ Dataset verification complete.")

In [ ]:
# ============================================================
# MODEL 1 — TF-IDF + LOGISTIC REGRESSION
# CELL 4: TF-IDF FEATURE EXTRACTION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Text and labels
X_train_text = train_df["comment"].astype(str)
X_val_text   = val_df["comment"].astype(str)
X_test_text  = test_df["comment"].astype(str)

y_train = train_df["sentiment"]
y_val   = val_df["sentiment"]
y_test  = test_df["sentiment"]

# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

# IMPORTANT:
# Fit only on training data
X_train_tfidf = tfidf.fit_transform(X_train_text)

# Transform validation and test using the fitted vectorizer
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("=" * 70)
print("TF-IDF FEATURES")
print("=" * 70)

print(f"Training matrix:   {X_train_tfidf.shape}")
print(f"Validation matrix: {X_val_tfidf.shape}")
print(f"Test matrix:       {X_test_tfidf.shape}")

print(f"\nVocabulary size: {len(tfidf.vocabulary_):,}")

print("\n✓ TF-IDF fitted only on training data.")
print("✓ Validation and test data transformed using the training vocabulary.")

In [ ]:
# ============================================================
# MODEL 1 — TF-IDF + LOGISTIC REGRESSION
# CELL 5: TRAIN MODEL
# ============================================================

from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Train Logistic Regression
# ------------------------------------------------------------

lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="liblinear",
    random_state=42
)

lr_model.fit(
    X_train_tfidf,
    y_train
)

print("=" * 70)
print("LOGISTIC REGRESSION TRAINING COMPLETE")
print("=" * 70)

print("Model: TF-IDF + Logistic Regression")
print(f"Training samples: {X_train_tfidf.shape[0]:,}")
print(f"TF-IDF features:  {X_train_tfidf.shape[1]:,}")

print("\n✓ Model trained successfully.")

In [ ]:
# ============================================================
# MODEL 1 — LOGISTIC REGRESSION
# CELL 6: VALIDATION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

y_val_pred_lr = lr_model.predict(X_val_tfidf)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

lr_val_accuracy = accuracy_score(y_val, y_val_pred_lr)

lr_val_precision = precision_score(
    y_val,
    y_val_pred_lr,
    average="macro",
    zero_division=0
)

lr_val_recall = recall_score(
    y_val,
    y_val_pred_lr,
    average="macro",
    zero_division=0
)

lr_val_f1_macro = f1_score(
    y_val,
    y_val_pred_lr,
    average="macro",
    zero_division=0
)

lr_val_f1_weighted = f1_score(
    y_val,
    y_val_pred_lr,
    average="weighted",
    zero_division=0
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("LOGISTIC REGRESSION — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:          {lr_val_accuracy:.4f}")
print(f"Macro Precision:   {lr_val_precision:.4f}")
print(f"Macro Recall:      {lr_val_recall:.4f}")
print(f"Macro F1:          {lr_val_f1_macro:.4f}")
print(f"Weighted F1:       {lr_val_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_lr,
        labels=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_lr = confusion_matrix(
    y_val,
    y_val_pred_lr,
    labels=["Negative", "Neutral", "Positive"]
)

cm_lr_df = pd.DataFrame(
    cm_lr,
    index=["Actual Negative", "Actual Neutral", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Neutral", "Predicted Positive"]
)

print("\nConfusion Matrix:")
display(cm_lr_df)

In [ ]:
# ============================================================
# MODEL 2 — TF-IDF + LINEAR SVM
# CELL 7: TRAIN MODEL
# ============================================================

from sklearn.svm import LinearSVC

# ------------------------------------------------------------
# Train Linear SVM
# ------------------------------------------------------------

svm_model = LinearSVC(
    C=1.0,
    random_state=42
)

svm_model.fit(
    X_train_tfidf,
    y_train
)

print("=" * 70)
print("LINEAR SVM TRAINING COMPLETE")
print("=" * 70)

print("Model: TF-IDF + Linear SVM")
print(f"Training samples: {X_train_tfidf.shape[0]:,}")
print(f"TF-IDF features:  {X_train_tfidf.shape[1]:,}")

print("\n✓ Model trained successfully.")

In [ ]:
# ============================================================
# MODEL 2 — TF-IDF + LINEAR SVM
# CELL 8: VALIDATION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

y_val_pred_svm = svm_model.predict(X_val_tfidf)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

svm_val_accuracy = accuracy_score(
    y_val,
    y_val_pred_svm
)

svm_val_precision = precision_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_val_recall = recall_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_val_f1_macro = f1_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_val_f1_weighted = f1_score(
    y_val,
    y_val_pred_svm,
    average="weighted",
    zero_division=0
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("LINEAR SVM — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:          {svm_val_accuracy:.4f}")
print(f"Macro Precision:   {svm_val_precision:.4f}")
print(f"Macro Recall:      {svm_val_recall:.4f}")
print(f"Macro F1:          {svm_val_f1_macro:.4f}")
print(f"Weighted F1:       {svm_val_f1_weighted:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        y_val_pred_svm,
        labels=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_svm = confusion_matrix(
    y_val,
    y_val_pred_svm,
    labels=["Negative", "Neutral", "Positive"]
)

cm_svm_df = pd.DataFrame(
    cm_svm,
    index=[
        "Actual Negative",
        "Actual Neutral",
        "Actual Positive"
    ],
    columns=[
        "Predicted Negative",
        "Predicted Neutral",
        "Predicted Positive"
    ]
)

print("\nConfusion Matrix:")
display(cm_svm_df)

In [ ]:
# Cell 9 — Check TensorFlow and GPU

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("✓ GPU is available")
else:
    print("⚠ No GPU detected — CNN/BERT training may be slow")

In [ ]:
# Cell 10 — Prepare text sequences for CNN

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# CNN vocabulary settings
MAX_VOCAB_SIZE = 50000
MAX_SEQUENCE_LENGTH = 150

cnn_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    oov_token="<OOV>"
)

# Fit ONLY on training comments
cnn_tokenizer.fit_on_texts(train_df["comment"].astype(str))

# Convert text to integer sequences
X_train_seq = cnn_tokenizer.texts_to_sequences(train_df["comment"].astype(str))
X_val_seq = cnn_tokenizer.texts_to_sequences(val_df["comment"].astype(str))

# Pad/truncate sequences
X_train_cnn = pad_sequences(
    X_train_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_cnn = pad_sequences(
    X_val_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

# Encode labels
label_to_id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

y_train_cnn = train_df["sentiment"].map(label_to_id).values
y_val_cnn = val_df["sentiment"].map(label_to_id).values

vocab_size = min(
    MAX_VOCAB_SIZE,
    len(cnn_tokenizer.word_index) + 1
)

print("=" * 70)
print("CNN TEXT REPRESENTATION")
print("=" * 70)
print("Training sequences:", X_train_cnn.shape)
print("Validation sequences:", X_val_cnn.shape)
print("Vocabulary size:", vocab_size)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Number of classes:", len(label_to_id))

In [ ]:
# Cell 11 — Build and compile CNN sentiment classifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

cnn_model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(3, activation="softmax")
])

cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

In [ ]:
# Cell 12 — Train CNN with early stopping

from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

cnn_history = cnn_model.fit(
    X_train_cnn,
    y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=10,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Cell 13 — Evaluate CNN on validation set

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Generate predictions using the restored best weights
y_val_pred_cnn_probs = cnn_model.predict(
    X_val_cnn,
    batch_size=256,
    verbose=1
)

y_val_pred_cnn = np.argmax(
    y_val_pred_cnn_probs,
    axis=1
)

# Convert numeric predictions back to sentiment labels
id_to_label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

y_val_pred_cnn_labels = pd.Series(y_val_pred_cnn).map(id_to_label)

# Calculate metrics
cnn_val_accuracy = accuracy_score(
    y_val,
    y_val_pred_cnn_labels
)

cnn_val_precision = precision_score(
    y_val,
    y_val_pred_cnn_labels,
    average="macro",
    zero_division=0
)

cnn_val_recall = recall_score(
    y_val,
    y_val_pred_cnn_labels,
    average="macro",
    zero_division=0
)

cnn_val_f1_macro = f1_score(
    y_val,
    y_val_pred_cnn_labels,
    average="macro",
    zero_division=0
)

cnn_val_f1_weighted = f1_score(
    y_val,
    y_val_pred_cnn_labels,
    average="weighted",
    zero_division=0
)

print("=" * 70)
print("CNN — VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:          {cnn_val_accuracy:.4f}")
print(f"Macro Precision:   {cnn_val_precision:.4f}")
print(f"Macro Recall:      {cnn_val_recall:.4f}")
print(f"Macro F1:          {cnn_val_f1_macro:.4f}")
print(f"Weighted F1:       {cnn_val_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_cnn_labels,
        labels=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

cm_cnn = confusion_matrix(
    y_val,
    y_val_pred_cnn_labels,
    labels=["Negative", "Neutral", "Positive"]
)

print("\nConfusion Matrix:")
print(cm_cnn)

In [ ]:
# Cell 14 — Check BERT / Hugging Face environment

import sys
import transformers
import torch

print("=" * 70)
print("BERT ENVIRONMENT")
print("=" * 70)

print("Python version:", sys.version.split()[0])
print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available — BERT will run on CPU")

In [ ]:
# Cell 15 — Load BERT tokenizer

from transformers import AutoTokenizer

BERT_MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_NAME
)

print("=" * 70)
print("BERT TOKENIZER")
print("=" * 70)
print("Model:", BERT_MODEL_NAME)
print("Vocabulary size:", bert_tokenizer.vocab_size)
print("Maximum model length:", bert_tokenizer.model_max_length)
print("Tokenizer loaded successfully.")

In [ ]:
# Cell 16 — Test BERT tokenization

import time

BERT_MAX_LENGTH = 128

# Use a small sample only for the benchmark
benchmark_texts = train_df["comment"].astype(str).iloc[:1000].tolist()

start_time = time.time()

benchmark_tokens = bert_tokenizer(
    benchmark_texts,
    padding="max_length",
    truncation=True,
    max_length=BERT_MAX_LENGTH,
    return_tensors="pt"
)

elapsed = time.time() - start_time

print("=" * 70)
print("BERT TOKENIZATION BENCHMARK")
print("=" * 70)
print(f"Texts tokenized: {len(benchmark_texts):,}")
print(f"Maximum sequence length: {BERT_MAX_LENGTH}")
print(f"Input tensor shape: {benchmark_tokens['input_ids'].shape}")
print(f"Tokenization time: {elapsed:.2f} seconds")
print(f"Tokens per second: {len(benchmark_texts) / elapsed:.1f}")

In [ ]:
# Cell 17 — Benchmark one BERT training step on CPU

import time
import torch
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 3

bert_model_benchmark = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_LABELS
)

device = torch.device("cpu")
bert_model_benchmark.to(device)
bert_model_benchmark.train()

# Small benchmark batch
benchmark_batch_size = 8

benchmark_inputs = {
    key: value[:benchmark_batch_size].to(device)
    for key, value in benchmark_tokens.items()
}

benchmark_labels = torch.tensor(
    y_train_cnn[:benchmark_batch_size],
    dtype=torch.long
).to(device)

optimizer_benchmark = torch.optim.AdamW(
    bert_model_benchmark.parameters(),
    lr=2e-5
)

# Warm-up
outputs = bert_model_benchmark(
    **benchmark_inputs,
    labels=benchmark_labels
)

loss = outputs.loss
loss.backward()
optimizer_benchmark.step()
optimizer_benchmark.zero_grad()

# Timed step
start_time = time.time()

outputs = bert_model_benchmark(
    **benchmark_inputs,
    labels=benchmark_labels
)

loss = outputs.loss
loss.backward()
optimizer_benchmark.step()
optimizer_benchmark.zero_grad()

elapsed = time.time() - start_time

print("=" * 70)
print("BERT CPU TRAINING BENCHMARK")
print("=" * 70)
print(f"Batch size:        {benchmark_batch_size}")
print(f"Sequence length:   {BERT_MAX_LENGTH}")
print(f"Training step:     {elapsed:.2f} seconds")
print(f"Steps per minute:  {60 / elapsed:.2f}")
print(f"Model parameters:  {sum(p.numel() for p in bert_model_benchmark.parameters()):,}")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 18 — Verify GPU and load BERT model

import torch
from transformers import AutoModelForSequenceClassification

# Define BERT model name again
BERT_MODEL_NAME = "bert-base-uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("BERT GPU SETUP")
print("=" * 70)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=3
)

bert_model.to(device)

print("\nBERT model loaded successfully.")
print("Model device:", next(bert_model.parameters()).device)

In [ ]:
# Cell 19 — Verify dataset state after GPU runtime change

import os
import pandas as pd

TRAIN_PATH = "/content/reddit_train.csv"
VAL_PATH   = "/content/reddit_validation.csv"
TEST_PATH  = "/content/reddit_test.csv"

print("=" * 70)
print("BERT DATASET CHECK")
print("=" * 70)

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(f"{os.path.basename(path)}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

# Reload datasets if necessary
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\nDataset sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nLabel distribution:")
print("\nTrain:")
print(train_df["sentiment"].value_counts())

print("\nValidation:")
print(val_df["sentiment"].value_counts())

print("\nTest:")
print(test_df["sentiment"].value_counts())

In [ ]:
# Cell 20 — Prepare BERT datasets and tokenizer

from transformers import AutoTokenizer
from datasets import Dataset

BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LENGTH = 128

# Label mapping
label_to_id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id_to_label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

# Create lightweight Hugging Face datasets
train_bert_df = train_df[["comment", "sentiment"]].copy()
val_bert_df = val_df[["comment", "sentiment"]].copy()

train_bert_df["label"] = train_bert_df["sentiment"].map(label_to_id)
val_bert_df["label"] = val_bert_df["sentiment"].map(label_to_id)

train_bert_dataset = Dataset.from_pandas(
    train_bert_df[["comment", "label"]],
    preserve_index=False
)

val_bert_dataset = Dataset.from_pandas(
    val_bert_df[["comment", "label"]],
    preserve_index=False
)

# Load tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_NAME
)

def tokenize_bert_batch(batch):
    return bert_tokenizer(
        batch["comment"],
        truncation=True,
        max_length=BERT_MAX_LENGTH
    )

# Tokenize without padding
train_bert_dataset = train_bert_dataset.map(
    tokenize_bert_batch,
    batched=True,
    batch_size=1000,
    desc="Tokenizing training data"
)

val_bert_dataset = val_bert_dataset.map(
    tokenize_bert_batch,
    batched=True,
    batch_size=1000,
    desc="Tokenizing validation data"
)

# Remove original text column
train_bert_dataset = train_bert_dataset.remove_columns(["comment"])
val_bert_dataset = val_bert_dataset.remove_columns(["comment"])

print("=" * 70)
print("BERT DATASETS READY")
print("=" * 70)

print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))
print("Maximum sequence length:", BERT_MAX_LENGTH)
print("Label mapping:", label_to_id)

print("\nFirst tokenized example:")
print(train_bert_dataset[0])

In [ ]:
# Cell 21 — Configure BERT fine-tuning

from transformers import (
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# Dynamic padding
bert_data_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer,
    padding="longest"
)

# Training configuration
bert_training_args = TrainingArguments(
    output_dir="/content/bert_reddit_results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Efficiency
    fp16=True,

    # Logging
    logging_strategy="steps",
    logging_steps=200,

    # Reproducibility
    seed=42,

    # Keep only best checkpoint
    save_total_limit=1,

    report_to="none"
)

print("=" * 70)
print("BERT TRAINING CONFIGURATION")
print("=" * 70)
print("Model:", BERT_MODEL_NAME)
print("Max sequence length:", BERT_MAX_LENGTH)
print("Train batch size:", 16)
print("Validation batch size:", 32)
print("Learning rate:", 2e-5)
print("Epochs:", 3)
print("Weight decay:", 0.01)
print("Mixed precision (FP16):", True)
print("Early stopping: validation loss")

In [ ]:
# Cell 22 — Create BERT evaluation metrics and Trainer

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_bert_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=train_bert_dataset,
    eval_dataset=val_bert_dataset,
    processing_class=bert_tokenizer,
    data_collator=bert_data_collator,
    compute_metrics=compute_bert_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("=" * 70)
print("BERT TRAINER READY")
print("=" * 70)
print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))
print("Early stopping patience: 1 epoch")
print("Trainer created successfully.")

In [ ]:
# Cell 23 — Fine-tune BERT

print("=" * 70)
print("STARTING BERT FINE-TUNING")
print("=" * 70)
print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))
print()

bert_train_result = bert_trainer.train()

print("\n" + "=" * 70)
print("BERT TRAINING COMPLETE")
print("=" * 70)
print("Training completed successfully.")

In [ ]:
# Cell 24 — Final BERT validation evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

print("=" * 70)
print("EVALUATING BEST BERT CHECKPOINT")
print("=" * 70)

bert_eval_output = bert_trainer.predict(
    val_bert_dataset
)

bert_logits = bert_eval_output.predictions
bert_labels = bert_eval_output.label_ids

bert_pred_ids = np.argmax(
    bert_logits,
    axis=-1
)

bert_pred_labels = [
    id_to_label[int(x)]
    for x in bert_pred_ids
]

bert_true_labels = [
    id_to_label[int(x)]
    for x in bert_labels
]

bert_val_accuracy = accuracy_score(
    bert_true_labels,
    bert_pred_labels
)

bert_val_precision = precision_score(
    bert_true_labels,
    bert_pred_labels,
    labels=["Negative", "Neutral", "Positive"],
    average="macro",
    zero_division=0
)

bert_val_recall = recall_score(
    bert_true_labels,
    bert_pred_labels,
    labels=["Negative", "Neutral", "Positive"],
    average="macro",
    zero_division=0
)

bert_val_f1_macro = f1_score(
    bert_true_labels,
    bert_pred_labels,
    labels=["Negative", "Neutral", "Positive"],
    average="macro",
    zero_division=0
)

bert_val_f1_weighted = f1_score(
    bert_true_labels,
    bert_pred_labels,
    labels=["Negative", "Neutral", "Positive"],
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("BERT — FINAL VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:          {bert_val_accuracy:.4f}")
print(f"Macro Precision:   {bert_val_precision:.4f}")
print(f"Macro Recall:      {bert_val_recall:.4f}")
print(f"Macro F1:          {bert_val_f1_macro:.4f}")
print(f"Weighted F1:       {bert_val_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        bert_true_labels,
        bert_pred_labels,
        labels=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

cm_bert = confusion_matrix(
    bert_true_labels,
    bert_pred_labels,
    labels=["Negative", "Neutral", "Positive"]
)

print("\nConfusion Matrix:")
print(cm_bert)

In [ ]:
# Cell 25 — Prepare training-only corpus for topic discovery

import pandas as pd
import re

topic_train = train_df[["comment", "id"]].copy()

# Remove empty comments
topic_train["comment"] = topic_train["comment"].astype(str).str.strip()
topic_train = topic_train[
    topic_train["comment"].notna() &
    (topic_train["comment"] != "")
].copy()

# Remove obvious Reddit deletion markers
deletion_markers = {
    "[deleted]",
    "[removed]",
    "[removed by reddit]"
}

topic_train = topic_train[
    ~topic_train["comment"].str.lower().isin(deletion_markers)
].copy()

print("=" * 70)
print("TOPIC DISCOVERY CORPUS")
print("=" * 70)
print("Training comments available:", len(topic_train))
print("Unique comment texts:", topic_train["comment"].nunique())
print("Unique thread/post IDs:", topic_train["id"].nunique())

print("\nNo validation or test comments are included.")

In [ ]:
# Cell 26 — Remove obvious platform/media noise for topic discovery
#
# IMPORTANT:
# These filters affect ONLY the topic-discovery corpus.
# The sentiment datasets remain unchanged.

topic_discovery = topic_train.copy()

text_lower = topic_discovery["comment"].str.lower().str.strip()

# 1. Remove Reddit bot / moderator / video-save artifacts
bot_pattern = (
    r"(savevideo|save.?video|downloadvideo|remindmebot|"
    r"automoderator|auto.?moderator|mod.?bot|bot\b)"
)

bot_mask = text_lower.str.contains(
    bot_pattern,
    regex=True,
    na=False
)

# Keep a count for reporting
bot_removed = int(bot_mask.sum())

topic_discovery = topic_discovery.loc[~bot_mask].copy()


# 2. Remove comments consisting primarily of media-only markers
media_pattern = (
    r"^(gif|image|img|video|photo|picture|"
    r"\[gif\]|\[image\]|\[video\]|\[photo\])$"
)

media_mask = (
    topic_discovery["comment"]
    .str.lower()
    .str.strip()
    .str.contains(media_pattern, regex=True, na=False)
)

media_removed = int(media_mask.sum())
topic_discovery = topic_discovery.loc[~media_mask].copy()


# 3. Remove URL-only comments
url_mask = topic_discovery["comment"].str.strip().str.match(
    r"^(https?://\S+|www\.\S+)$",
    na=False
)

url_removed = int(url_mask.sum())
topic_discovery = topic_discovery.loc[~url_mask].copy()


# 4. Remove symbol / emoji-only comments
def is_symbol_only(text):
    text = str(text).strip()
    if not text:
        return True

    # Keep comments containing at least one alphanumeric character
    return not bool(re.search(r"[A-Za-z0-9]", text))

symbol_mask = topic_discovery["comment"].apply(is_symbol_only)

symbol_removed = int(symbol_mask.sum())
topic_discovery = topic_discovery.loc[~symbol_mask].copy()


# 5. Remove very short generic reactions
generic_reactions = {
    "lol",
    "lmao",
    "haha",
    "hahaha",
    "wow",
    "good",
    "great",
    "nice",
    "beautiful",
    "hero",
    "rip",
    "r.i.p",
    "ok",
    "okay",
    "yes",
    "no",
    "thanks",
    "thank you",
    "exactly"
}

generic_mask = (
    topic_discovery["comment"]
    .str.lower()
    .str.strip()
    .isin(generic_reactions)
)

generic_removed = int(generic_mask.sum())
topic_discovery = topic_discovery.loc[~generic_mask].copy()


# 6. Remove remaining Reddit deletion marker
remaining_marker_mask = (
    topic_discovery["comment"]
    .str.lower()
    .str.strip()
    == "[removed by reddit]"
)

marker_removed = int(remaining_marker_mask.sum())
topic_discovery = topic_discovery.loc[~remaining_marker_mask].copy()


print("=" * 70)
print("FILTERED TOPIC-DISCOVERY CORPUS")
print("=" * 70)

print("Original training comments:", len(topic_train))
print("Bot/moderator artifacts removed:", bot_removed)
print("Media-only comments removed:", media_removed)
print("URL-only comments removed:", url_removed)
print("Symbol/emoji-only comments removed:", symbol_removed)
print("Generic short reactions removed:", generic_removed)
print("Remaining deletion markers removed:", marker_removed)

print("\nFinal topic-discovery comments:", len(topic_discovery))
print("Unique comment texts:", topic_discovery["comment"].nunique())
print("Unique thread/post IDs:", topic_discovery["id"].nunique())

In [ ]:
# Cell 27 — Load sentence embedding model for topic discovery

from sentence_transformers import SentenceTransformer
import numpy as np

TOPIC_EMBEDDING_MODEL = "all-MiniLM-L6-v2"

topic_encoder = SentenceTransformer(
    TOPIC_EMBEDDING_MODEL,
    device="cuda"
)

print("=" * 70)
print("TOPIC EMBEDDING MODEL")
print("=" * 70)
print("Model:", TOPIC_EMBEDDING_MODEL)
print("Device:", topic_encoder.device)
print("Embedding dimension:", topic_encoder.get_sentence_embedding_dimension())
print("Comments to embed:", len(topic_discovery))

In [ ]:
# Cell 28 — Generate semantic embeddings for topic discovery

import numpy as np
import time

topic_texts = topic_discovery["comment"].astype(str).tolist()

start_time = time.time()

topic_embeddings = topic_encoder.encode(
    topic_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time

print("=" * 70)
print("TOPIC EMBEDDINGS CREATED")
print("=" * 70)
print("Number of comments:", len(topic_texts))
print("Embedding shape:", topic_embeddings.shape)
print("Embedding dimension:", topic_embeddings.shape[1])
print(f"Embedding time: {elapsed / 60:.2f} minutes")
print("Normalized embeddings:", True)
print("NaN values:", np.isnan(topic_embeddings).sum())

In [ ]:
# Cell 29 — Discover 10 semantic topics using K-Means

from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

N_TOPICS = 10

topic_kmeans = KMeans(
    n_clusters=N_TOPICS,
    random_state=42,
    n_init=10
)

topic_cluster_ids = topic_kmeans.fit_predict(
    topic_embeddings
)

# Store cluster assignments
topic_discovery = topic_discovery.copy()
topic_discovery["topic_cluster"] = topic_cluster_ids

print("=" * 70)
print("10-CLUSTER TOPIC DISCOVERY")
print("=" * 70)

print("Number of clusters:", N_TOPICS)
print("Comments clustered:", len(topic_discovery))
print("Embedding dimension:", topic_embeddings.shape[1])

print("\nCluster sizes:")
cluster_counts = (
    topic_discovery["topic_cluster"]
    .value_counts()
    .sort_index()
)

for cluster_id, count in cluster_counts.items():
    percentage = count / len(topic_discovery) * 100
    print(
        f"Cluster {cluster_id}: "
        f"{count:,} comments ({percentage:.2f}%)"
    )

print("\nK-Means clustering completed successfully.")

In [ ]:
# Cell 30 — Inspect representative comments from each topic cluster

import numpy as np
import pandas as pd

# Calculate cosine similarity to each cluster centroid.
# Since embeddings are normalized, dot product gives cosine similarity.
cluster_centers = topic_kmeans.cluster_centers_
cluster_centers = cluster_centers / np.linalg.norm(
    cluster_centers,
    axis=1,
    keepdims=True
)

similarity_matrix = topic_embeddings @ cluster_centers.T

# Show the 8 most representative comments per cluster
N_EXAMPLES = 8

print("=" * 80)
print("REPRESENTATIVE COMMENTS FOR 10 TOPIC CLUSTERS")
print("=" * 80)

for cluster_id in range(N_TOPICS):

    cluster_indices = np.where(
        topic_cluster_ids == cluster_id
    )[0]

    # Rank comments by similarity to their own cluster centroid
    similarities = similarity_matrix[
        cluster_indices,
        cluster_id
    ]

    top_positions = np.argsort(similarities)[
        ::-1
    ][:N_EXAMPLES]

    selected_indices = cluster_indices[top_positions]

    print("\n" + "=" * 80)
    print(
        f"CLUSTER {cluster_id} "
        f"({len(cluster_indices):,} comments)"
    )
    print("=" * 80)

    for rank, idx in enumerate(selected_indices, start=1):
        comment = topic_discovery.iloc[idx]["comment"]

        # Keep very long comments readable
        comment_display = str(comment).replace("\n", " ").strip()

        if len(comment_display) > 300:
            comment_display = comment_display[:300] + "..."

        print(f"{rank}. {comment_display}")

In [ ]:
# Cell 31 — Prepare normalized topic centroids

import numpy as np

# Normalize K-Means centroids for cosine similarity
topic_centroids = topic_kmeans.cluster_centers_.copy()

topic_centroids = topic_centroids / np.linalg.norm(
    topic_centroids,
    axis=1,
    keepdims=True
)

print("=" * 70)
print("TOPIC CENTROIDS")
print("=" * 70)
print("Number of topics:", topic_centroids.shape[0])
print("Centroid dimension:", topic_centroids.shape[1])
print("Centroids normalized:", True)
print("NaN values:", np.isnan(topic_centroids).sum())

In [ ]:
# Cell 32 — Assign frozen topic IDs to train and validation comments

import numpy as np
import pandas as pd
import time

def assign_frozen_topics(texts, encoder, centroids, batch_size=128):
    """
    Encode texts and assign each comment to the nearest frozen
    topic centroid using cosine similarity.
    """

    embeddings = encoder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = embeddings @ centroids.T

    topic_ids = np.argmax(
        similarities,
        axis=1
    )

    topic_confidence = np.max(
        similarities,
        axis=1
    )

    return topic_ids, topic_confidence


print("=" * 70)
print("ASSIGNING FROZEN TOPICS")
print("=" * 70)

# Train
start_time = time.time()

train_topic_ids, train_topic_confidence = assign_frozen_topics(
    train_df["comment"].astype(str).tolist(),
    topic_encoder,
    topic_centroids
)

print(f"\nTraining topic assignment time: {(time.time() - start_time) / 60:.2f} minutes")

# Validation
start_time = time.time()

val_topic_ids, val_topic_confidence = assign_frozen_topics(
    val_df["comment"].astype(str).tolist(),
    topic_encoder,
    topic_centroids
)

print(f"Validation topic assignment time: {(time.time() - start_time) / 60:.2f} minutes")

# Store assignments
train_df["topic_id"] = train_topic_ids
train_df["topic_confidence"] = train_topic_confidence

val_df["topic_id"] = val_topic_ids
val_df["topic_confidence"] = val_topic_confidence

print("\n" + "=" * 70)
print("TOPIC ASSIGNMENT COMPLETE")
print("=" * 70)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print("\nTrain topic distribution:")
print(
    train_df["topic_id"]
    .value_counts()
    .sort_index()
)

print("\nValidation topic distribution:")
print(
    val_df["topic_id"]
    .value_counts()
    .sort_index()
)

print("\nAverage topic confidence:")
print(f"Train:      {train_df['topic_confidence'].mean():.4f}")
print(f"Validation: {val_df['topic_confidence'].mean():.4f}")

In [ ]:
# Cell 33 — Add frozen topic IDs to BERT datasets

# Add topic IDs to the existing BERT datasets
train_bert_dataset = train_bert_dataset.add_column(
    "topic_id",
    train_df["topic_id"].astype(int).tolist()
)

val_bert_dataset = val_bert_dataset.add_column(
    "topic_id",
    val_df["topic_id"].astype(int).tolist()
)

print("=" * 70)
print("BERT-CNN TOPIC FEATURE")
print("=" * 70)

print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))

print("\nTraining dataset features:")
print(train_bert_dataset.features)

print("\nFirst training example:")
print(train_bert_dataset[0])

In [ ]:
# Cell 34 — Define BERT-CNN + Topic model

import torch
import torch.nn as nn
from transformers import BertModel

class BertCNNTopicClassifier(nn.Module):
    def __init__(
        self,
        model_name="bert-base-uncased",
        num_topics=10,
        topic_embedding_dim=16,
        num_filters=128,
        kernel_size=5,
        dense_dim=128,
        num_labels=3,
        dropout=0.3
    ):
        super().__init__()

        # Pretrained BERT encoder
        self.bert = BertModel.from_pretrained(model_name)

        hidden_size = self.bert.config.hidden_size

        # CNN over BERT token representations
        self.conv1d = nn.Conv1d(
            in_channels=hidden_size,
            out_channels=num_filters,
            kernel_size=kernel_size
        )

        self.global_max_pool = nn.AdaptiveMaxPool1d(1)

        # Learnable topic representation
        self.topic_embedding = nn.Embedding(
            num_embeddings=num_topics,
            embedding_dim=topic_embedding_dim
        )

        # Combine CNN text representation + topic representation
        self.classifier = nn.Sequential(
            nn.Linear(
                num_filters + topic_embedding_dim,
                dense_dim
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dense_dim, num_labels)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        topic_id=None,
        labels=None
    ):
        # BERT
        bert_outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Last hidden state:
        # [batch_size, sequence_length, hidden_size]
        sequence_output = bert_outputs.last_hidden_state

        # CNN expects:
        # [batch_size, hidden_size, sequence_length]
        sequence_output = sequence_output.transpose(1, 2)

        # Convolution
        cnn_output = self.conv1d(sequence_output)
        cnn_output = torch.relu(cnn_output)

        # Global max pooling
        pooled_output = self.global_max_pool(
            cnn_output
        ).squeeze(-1)

        # Topic embedding
        topic_output = self.topic_embedding(topic_id)

        # Concatenate text + topic
        combined = torch.cat(
            [pooled_output, topic_output],
            dim=1
        )

        # Classification
        logits = self.classifier(combined)

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }


bert_cnn_topic_model = BertCNNTopicClassifier(
    model_name=BERT_MODEL_NAME,
    num_topics=10,
    topic_embedding_dim=16,
    num_filters=128,
    kernel_size=5,
    dense_dim=128,
    num_labels=3,
    dropout=0.3
)

bert_cnn_topic_model.to(device)

print("=" * 70)
print("BERT-CNN + TOPIC MODEL")
print("=" * 70)

print("BERT model:", BERT_MODEL_NAME)
print("BERT hidden size:", bert_cnn_topic_model.bert.config.hidden_size)
print("CNN filters:", 128)
print("CNN kernel size:", 5)
print("Topic embedding dimension:", 16)
print("Number of topics:", 10)
print("Dense dimension:", 128)
print("Number of classes:", 3)

total_params = sum(
    p.numel()
    for p in bert_cnn_topic_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in bert_cnn_topic_model.parameters()
    if p.requires_grad
)

print("\nTotal parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Model device:", next(bert_cnn_topic_model.parameters()).device)

In [ ]:
# Cell 35 — Data collator for BERT-CNN + Topic

from transformers import DataCollatorWithPadding

base_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer,
    padding="longest"
)

def bert_cnn_topic_collator(features):
    # Extract topic IDs before passing the remaining
    # BERT features to the standard padding collator
    topic_ids = torch.tensor(
        [feature["topic_id"] for feature in features],
        dtype=torch.long
    )

    features_without_topic = [
        {
            key: value
            for key, value in feature.items()
            if key != "topic_id"
        }
        for feature in features
    ]

    batch = base_collator(features_without_topic)

    # Add topic feature back to the batch
    batch["topic_id"] = topic_ids

    return batch

# Test the collator on a small batch
test_features = [
    train_bert_dataset[i]
    for i in range(4)
]

test_batch = bert_cnn_topic_collator(test_features)

print("=" * 70)
print("BERT-CNN DATA COLLATOR")
print("=" * 70)

for key, value in test_batch.items():
    print(
        f"{key}: shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

In [ ]:
# Cell 36 — Evaluation metrics for BERT-CNN + Topic

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_bert_cnn_metrics(eval_pred):
    logits, labels = eval_pred

    # Handle models that return tuples
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

print("Metric function ready.")

In [ ]:
# Cell 37 — Training configuration for BERT-CNN + Topic

from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

BERT_CNN_OUTPUT_DIR = "/content/bert_cnn_topic_results"

bert_cnn_training_args = TrainingArguments(
    output_dir=BERT_CNN_OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,

    # Performance
    fp16=True,

    # Reproducibility
    seed=42,

    # Logging
    logging_strategy="epoch",
    report_to="none",

    # Our model explicitly accepts topic_id
    remove_unused_columns=False
)

print("=" * 70)
print("BERT-CNN + TOPIC TRAINING CONFIGURATION")
print("=" * 70)
print("Epochs:", bert_cnn_training_args.num_train_epochs)
print("Train batch size:", bert_cnn_training_args.per_device_train_batch_size)
print("Eval batch size:", bert_cnn_training_args.per_device_eval_batch_size)
print("Learning rate:", bert_cnn_training_args.learning_rate)
print("Weight decay:", bert_cnn_training_args.weight_decay)
print("FP16:", bert_cnn_training_args.fp16)
print("Best model metric:", bert_cnn_training_args.metric_for_best_model)
print("Seed:", bert_cnn_training_args.seed)
print("Test set used:", False)

In [ ]:
# Cell 38 — Create Trainer for BERT-CNN + Topic

bert_cnn_trainer = Trainer(
    model=bert_cnn_topic_model,
    args=bert_cnn_training_args,

    train_dataset=train_bert_dataset,
    eval_dataset=val_bert_dataset,

    data_collator=bert_cnn_topic_collator,
    compute_metrics=compute_bert_cnn_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("=" * 70)
print("BERT-CNN + TOPIC TRAINER")
print("=" * 70)
print("Trainer created successfully.")
print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))
print("Test examples used:", 0)

In [ ]:
# Cell 39 — Train BERT-CNN + Topic

print("=" * 70)
print("STARTING BERT-CNN + TOPIC TRAINING")
print("=" * 70)
print("Training examples:", len(train_bert_dataset))
print("Validation examples:", len(val_bert_dataset))
print("Test set: NOT USED")
print("\nTraining...")

bert_cnn_train_result = bert_cnn_trainer.train()

print("\n" + "=" * 70)
print("BERT-CNN + TOPIC TRAINING COMPLETE")
print("=" * 70)

print(
    "Training runtime (seconds):",
    round(bert_cnn_train_result.metrics["train_runtime"], 2)
)

print(
    "Training samples/second:",
    round(bert_cnn_train_result.metrics["train_samples_per_second"], 2)
)

print(
    "Final reported training loss:",
    round(bert_cnn_train_result.metrics["train_loss"], 4)
)

In [ ]:
# Cell 40 — Final validation evaluation of the restored BERT-CNN + Topic model

print("=" * 70)
print("FINAL VALIDATION EVALUATION — BERT-CNN + TOPIC")
print("=" * 70)

bert_cnn_val_results = bert_cnn_trainer.evaluate(
    eval_dataset=val_bert_dataset
)

print("\nValidation results:")

for key, value in bert_cnn_val_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Cell 41 — Detailed validation report for BERT-CNN + Topic

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("DETAILED VALIDATION REPORT — BERT-CNN + TOPIC")
print("=" * 70)

# Get predictions on validation set
val_predictions_output = bert_cnn_trainer.predict(
    val_bert_dataset
)

val_logits = val_predictions_output.predictions

if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

val_pred_labels = np.argmax(val_logits, axis=-1)
val_true_labels = np.array(val_bert_dataset["label"])

label_names = ["Negative", "Neutral", "Positive"]

print("\nClassification Report:")
print(
    classification_report(
        val_true_labels,
        val_pred_labels,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    val_true_labels,
    val_pred_labels
)

print("Confusion Matrix:")
print(cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("Label order:", label_names)

In [ ]:
# Cell 42 — Consolidate validation results for all five models

validation_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": 0.8139,
        "Macro Precision": 0.8146,
        "Macro Recall": 0.8143,
        "Macro F1": 0.8140,
        "Weighted F1": 0.8137
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Accuracy": 0.8390,
        "Macro Precision": 0.8392,
        "Macro Recall": 0.8395,
        "Macro F1": 0.8393,
        "Weighted F1": 0.8390
    },
    {
        "Model": "CNN",
        "Accuracy": 0.8747,
        "Macro Precision": 0.8804,
        "Macro Recall": 0.8751,
        "Macro F1": 0.8764,
        "Weighted F1": 0.8754
    },
    {
        "Model": "BERT",
        "Accuracy": 0.9295,
        "Macro Precision": 0.9307,
        "Macro Recall": 0.9294,
        "Macro F1": 0.9298,
        "Weighted F1": 0.9296
    },
    {
        "Model": "BERT-CNN + Topic",
        "Accuracy": 0.9201,
        "Macro Precision": 0.9226,
        "Macro Recall": 0.9205,
        "Macro F1": 0.9208,
        "Weighted F1": 0.9203
    }
])

print("=" * 70)
print("VALIDATION RESULTS — FIVE-MODEL COMPARISON")
print("=" * 70)

display(
    validation_results.style.format({
        "Accuracy": "{:.4f}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}",
        "Macro F1": "{:.4f}",
        "Weighted F1": "{:.4f}"
    })
)

In [8]:
# Cell 45 — Safely locate the existing fitted KMeans object

from sklearn.cluster import KMeans

print("=" * 70)
print("LOCATING FROZEN KMEANS MODEL")
print("=" * 70)

kmeans_candidates = []

# Take a snapshot before iterating
global_items = list(globals().items())

for name, obj in global_items:
    try:
        is_kmeans = isinstance(obj, KMeans)
    except Exception:
        is_kmeans = False

    if is_kmeans and hasattr(obj, "cluster_centers_"):
        kmeans_candidates.append(name)

print("KMeans objects found:", kmeans_candidates)

for name in kmeans_candidates:
    obj = globals()[name]
    print(
        f"{name}: "
        f"clusters={obj.n_clusters}, "
        f"centroid_shape={obj.cluster_centers_.shape}"
    )

LOCATING FROZEN KMEANS MODEL
KMeans objects found: []


In [9]:
# Cell 46 — Assign frozen training-derived topics to the held-out test set

from sklearn.preprocessing import normalize

print("=" * 70)
print("ASSIGNING FROZEN TOPICS — HELD-OUT TEST SET")
print("=" * 70)

# Normalize the frozen training-derived centroids
frozen_centroids = normalize(
    topic_kmeans.cluster_centers_
)

# Cosine similarity between each test embedding
# and each frozen training-derived centroid
test_similarities = np.matmul(
    test_embeddings,
    frozen_centroids.T
)

# Assign the nearest frozen topic
test_topic_ids = np.argmax(
    test_similarities,
    axis=1
)

# Maximum cosine similarity = assignment confidence
test_topic_confidence = np.max(
    test_similarities,
    axis=1
)

# Add topic information to the test dataframe
test_df["topic_id"] = test_topic_ids.astype(int)
test_df["topic_confidence"] = test_topic_confidence.astype(float)

print("Test topic assignment complete.")
print("\nTest topic distribution:")

topic_distribution = (
    test_df["topic_id"]
    .value_counts()
    .sort_index()
)

for topic_id, count in topic_distribution.items():
    print(
        f"Topic {topic_id}: "
        f"{count:5d} "
        f"({count / len(test_df) * 100:6.2f}%)"
    )

print(
    "\nMean topic confidence:",
    round(test_df["topic_confidence"].mean(), 4)
)

print(
    "Minimum topic confidence:",
    round(test_df["topic_confidence"].min(), 4)
)

print(
    "Maximum topic confidence:",
    round(test_df["topic_confidence"].max(), 4)
)

print("\nTest rows:", len(test_df))
print("Test labels available:", "sentiment" in test_df.columns)

ASSIGNING FROZEN TOPICS — HELD-OUT TEST SET


NameError: name 'topic_kmeans' is not defined

In [ ]:
# Cell 47 — Prepare held-out test dataset for BERT-CNN + Topic

from datasets import Dataset

print("=" * 70)
print("PREPARING HELD-OUT TEST DATASET")
print("=" * 70)

# Preserve the frozen test labels for final evaluation
test_labels = test_df["sentiment"].map({
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}).astype(int).tolist()

# Create Hugging Face dataset
test_bert_dataset = Dataset.from_pandas(
    test_df[["comment"]].copy(),
    preserve_index=False
)

# Tokenize using the same tokenizer/settings as training
def tokenize_test_batch(batch):
    return bert_tokenizer(
        batch["comment"],
        truncation=True,
        max_length=128
    )

test_bert_dataset = test_bert_dataset.map(
    tokenize_test_batch,
    batched=True,
    desc="Tokenizing test comments"
)

# Add labels
test_bert_dataset = test_bert_dataset.add_column(
    "label",
    test_labels
)

# Add frozen topic assignments
test_bert_dataset = test_bert_dataset.add_column(
    "topic_id",
    test_df["topic_id"].astype(int).tolist()
)

print("\nDataset prepared.")
print("Number of examples:", len(test_bert_dataset))
print("Features:")
print(test_bert_dataset.features)

print("\nFirst example:")
print("Label:", test_bert_dataset[0]["label"])
print("Topic ID:", test_bert_dataset[0]["topic_id"])
print("Token count:", len(test_bert_dataset[0]["input_ids"]))

In [ ]:
# Cell 48 — Final held-out test evaluation
# BERT-CNN + Topic

print("=" * 70)
print("FINAL TEST EVALUATION — BERT-CNN + TOPIC")
print("=" * 70)

bert_cnn_test_results = bert_cnn_trainer.predict(
    test_bert_dataset
)

test_logits = bert_cnn_test_results.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_pred_labels = np.argmax(
    test_logits,
    axis=-1
)

test_true_labels = np.array(
    test_bert_dataset["label"]
)

label_names = [
    "Negative",
    "Neutral",
    "Positive"
]

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        test_true_labels,
        test_pred_labels,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

# Confusion matrix
test_cm = confusion_matrix(
    test_true_labels,
    test_pred_labels
)

print("Confusion Matrix:")
print(test_cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("Label order:", label_names)

# Overall metrics
test_accuracy = accuracy_score(
    test_true_labels,
    test_pred_labels
)

test_precision, test_recall, test_f1_macro, _ = (
    precision_recall_fscore_support(
        test_true_labels,
        test_pred_labels,
        average="macro",
        zero_division=0
    )
)

_, _, test_f1_weighted, _ = (
    precision_recall_fscore_support(
        test_true_labels,
        test_pred_labels,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {test_accuracy:.4f}")
print(f"Macro Precision:  {test_precision:.4f}")
print(f"Macro Recall:     {test_recall:.4f}")
print(f"Macro F1:         {test_f1_macro:.4f}")
print(f"Weighted F1:      {test_f1_weighted:.4f}")

In [ ]:
# Cell 49 — Remove unused raw comment column from test dataset

print("=" * 70)
print("CLEANING TEST DATASET FOR MODEL INFERENCE")
print("=" * 70)

print("Features before:")
print(test_bert_dataset.features)

# Keep only model-relevant fields.
# The original text is retained separately in test_df.
test_bert_dataset = test_bert_dataset.remove_columns(
    ["comment"]
)

print("\nFeatures after:")
print(test_bert_dataset.features)

print("\nNumber of test examples:", len(test_bert_dataset))
print("Test labels preserved:", "label" in test_bert_dataset.column_names)
print("Test topics preserved:", "topic_id" in test_bert_dataset.column_names)

In [ ]:
# Cell 50 — Final held-out test evaluation
# BERT-CNN + Topic

print("=" * 70)
print("FINAL TEST EVALUATION — BERT-CNN + TOPIC")
print("=" * 70)

bert_cnn_test_results = bert_cnn_trainer.predict(
    test_bert_dataset
)

test_logits = bert_cnn_test_results.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_pred_labels = np.argmax(
    test_logits,
    axis=-1
)

test_true_labels = np.array(
    test_bert_dataset["label"]
)

label_names = [
    "Negative",
    "Neutral",
    "Positive"
]

print("\nClassification Report:")
print(
    classification_report(
        test_true_labels,
        test_pred_labels,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

test_cm = confusion_matrix(
    test_true_labels,
    test_pred_labels
)

print("Confusion Matrix:")
print(test_cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("Label order:", label_names)

test_accuracy = accuracy_score(
    test_true_labels,
    test_pred_labels
)

test_precision, test_recall, test_f1_macro, _ = (
    precision_recall_fscore_support(
        test_true_labels,
        test_pred_labels,
        average="macro",
        zero_division=0
    )
)

_, _, test_f1_weighted, _ = (
    precision_recall_fscore_support(
        test_true_labels,
        test_pred_labels,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {test_accuracy:.4f}")
print(f"Macro Precision:  {test_precision:.4f}")
print(f"Macro Recall:     {test_recall:.4f}")
print(f"Macro F1:         {test_f1_macro:.4f}")
print(f"Weighted F1:      {test_f1_weighted:.4f}")


In [ ]:
# Cell 51 — Final held-out test evaluation
# TF-IDF + Logistic Regression

print("=" * 70)
print("FINAL TEST EVALUATION — TF-IDF + LOGISTIC REGRESSION")
print("=" * 70)

# Transform test comments using the already-fitted training TF-IDF vocabulary.
X_test_tfidf = tfidf_vectorizer.transform(
    test_df["comment"].astype(str)
)

print("Test matrix shape:", X_test_tfidf.shape)

# Predict — no fitting occurs here.
logreg_test_pred = logreg_model.predict(
    X_test_tfidf
)

# Convert labels to the same numeric ordering used elsewhere
label_to_id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

logreg_test_true = (
    test_df["sentiment"]
    .map(label_to_id)
    .astype(int)
    .to_numpy()
)

print("\nClassification Report:")
print(
    classification_report(
        logreg_test_true,
        logreg_test_pred,
        target_names=["Negative", "Neutral", "Positive"],
        digits=4,
        zero_division=0
    )
)

logreg_test_cm = confusion_matrix(
    logreg_test_true,
    logreg_test_pred
)

print("Confusion Matrix:")
print(logreg_test_cm)

print("\nRows = Actual")
print("Columns = Predicted")

logreg_test_accuracy = accuracy_score(
    logreg_test_true,
    logreg_test_pred
)

logreg_test_precision, logreg_test_recall, logreg_test_f1, _ = (
    precision_recall_fscore_support(
        logreg_test_true,
        logreg_test_pred,
        average="macro",
        zero_division=0
    )
)

_, _, logreg_test_f1_weighted, _ = (
    precision_recall_fscore_support(
        logreg_test_true,
        logreg_test_pred,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {logreg_test_accuracy:.4f}")
print(f"Macro Precision:  {logreg_test_precision:.4f}")
print(f"Macro Recall:     {logreg_test_recall:.4f}")
print(f"Macro F1:         {logreg_test_f1:.4f}")
print(f"Weighted F1:      {logreg_test_f1_weighted:.4f}")

In [ ]:
# Cell 52 — Recover TF-IDF vectorizer from training data only

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 70)
print("RECOVERING TRAINING-ONLY TF-IDF VECTORIZER")
print("=" * 70)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

# IMPORTANT:
# Fit only on the frozen training comments.
X_train_tfidf_recovered = tfidf_vectorizer.fit_transform(
    train_df["comment"].astype(str)
)

print("Training matrix shape:", X_train_tfidf_recovered.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

print("\nValidation/test data used for fitting: False")

In [ ]:
# Cell 53 — Locate the previously fitted Logistic Regression model

from sklearn.linear_model import LogisticRegression

print("=" * 70)
print("LOCATING FITTED LOGISTIC REGRESSION MODEL")
print("=" * 70)

logreg_candidates = []

for name, obj in list(globals().items()):
    try:
        is_logreg = isinstance(obj, LogisticRegression)
    except Exception:
        is_logreg = False

    if is_logreg and hasattr(obj, "coef_"):
        logreg_candidates.append(name)

print("Fitted Logistic Regression objects found:", logreg_candidates)

for name in logreg_candidates:
    obj = globals()[name]
    print(
        f"{name}: "
        f"classes={obj.classes_}, "
        f"coef_shape={obj.coef_.shape}, "
        f"C={obj.C}"
    )


In [ ]:
# Cell 54 — Check for saved model/vectorizer artifacts

import os

print("=" * 70)
print("CHECKING FOR SAVED CLASSICAL MODEL ARTIFACTS")
print("=" * 70)

search_paths = [
    "/content",
    "/content/bert_cnn_topic_results"
]

possible_files = []

for base_path in search_paths:
    if os.path.exists(base_path):
        for root, dirs, files in os.walk(base_path):
            for filename in files:
                lower_name = filename.lower()

                if any(
                    keyword in lower_name
                    for keyword in [
                        "logreg",
                        "logistic",
                        "tfidf",
                        "svm",
                        "linear"
                    ]
                ):
                    possible_files.append(
                        os.path.join(root, filename)
                    )

if possible_files:
    print("Potential saved artifacts found:")
    for path in sorted(possible_files):
        print(" -", path)
else:
    print("No saved Logistic Regression/TF-IDF/SVM artifacts found.")

In [ ]:
# Cell 55 — Reconstruct the previously trained Logistic Regression model
# using training data only

from sklearn.linear_model import LogisticRegression

print("=" * 70)
print("RECONSTRUCTING TF-IDF + LOGISTIC REGRESSION")
print("=" * 70)

# Use the recovered training-only TF-IDF matrix
logreg_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="liblinear",
    random_state=42
)

train_labels_logreg = (
    train_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("Training examples:", X_train_tfidf_recovered.shape[0])
print("TF-IDF features:", X_train_tfidf_recovered.shape[1])
print("Fitting Logistic Regression...")
print("Validation/test data used for fitting: False")

logreg_model.fit(
    X_train_tfidf_recovered,
    train_labels_logreg
)

print("\nReconstruction complete.")
print("Classes:", logreg_model.classes_)
print("Coefficient shape:", logreg_model.coef_.shape)
print("Intercept shape:", logreg_model.intercept_.shape)

In [ ]:
# Cell 56 — Final held-out test evaluation
# TF-IDF + Logistic Regression

print("=" * 70)
print("FINAL TEST EVALUATION — TF-IDF + LOGISTIC REGRESSION")
print("=" * 70)

# Transform test comments using the frozen training-only vocabulary.
X_test_tfidf = tfidf_vectorizer.transform(
    test_df["comment"].astype(str)
)

print("Test matrix shape:", X_test_tfidf.shape)

# Predict using the reconstructed frozen model.
logreg_test_pred = logreg_model.predict(
    X_test_tfidf
)

logreg_test_true = (
    test_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("\nClassification Report:")
print(
    classification_report(
        logreg_test_true,
        logreg_test_pred,
        target_names=["Negative", "Neutral", "Positive"],
        digits=4,
        zero_division=0
    )
)

logreg_test_cm = confusion_matrix(
    logreg_test_true,
    logreg_test_pred
)

print("Confusion Matrix:")
print(logreg_test_cm)

print("\nRows = Actual")
print("Columns = Predicted")

logreg_test_accuracy = accuracy_score(
    logreg_test_true,
    logreg_test_pred
)

logreg_test_precision, logreg_test_recall, logreg_test_f1, _ = (
    precision_recall_fscore_support(
        logreg_test_true,
        logreg_test_pred,
        average="macro",
        zero_division=0
    )
)

_, _, logreg_test_f1_weighted, _ = (
    precision_recall_fscore_support(
        logreg_test_true,
        logreg_test_pred,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {logreg_test_accuracy:.4f}")
print(f"Macro Precision:  {logreg_test_precision:.4f}")
print(f"Macro Recall:     {logreg_test_recall:.4f}")
print(f"Macro F1:         {logreg_test_f1:.4f}")
print(f"Weighted F1:      {logreg_test_f1_weighted:.4f}")

In [ ]:
# Cell 57 — Locate the previously fitted Linear SVM

from sklearn.svm import LinearSVC

print("=" * 70)
print("LOCATING FITTED LINEAR SVM")
print("=" * 70)

svm_candidates = []

for name, obj in list(globals().items()):
    try:
        is_svm = isinstance(obj, LinearSVC)
    except Exception:
        is_svm = False

    if is_svm and hasattr(obj, "coef_"):
        svm_candidates.append(name)

print("Fitted Linear SVM objects found:", svm_candidates)

for name in svm_candidates:
    obj = globals()[name]
    print(
        f"{name}: "
        f"classes={obj.classes_}, "
        f"coef_shape={obj.coef_.shape}, "
        f"C={obj.C}"
    )

In [ ]:
# Cell 58 — Reconstruct the previously trained Linear SVM
# using training data only

from sklearn.svm import LinearSVC

print("=" * 70)
print("RECONSTRUCTING TF-IDF + LINEAR SVM")
print("=" * 70)

svm_model = LinearSVC(
    C=1.0,
    random_state=42
)

train_labels_svm = (
    train_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("Training examples:", X_train_tfidf_recovered.shape[0])
print("TF-IDF features:", X_train_tfidf_recovered.shape[1])
print("Fitting Linear SVM...")
print("Validation/test data used for fitting: False")

svm_model.fit(
    X_train_tfidf_recovered,
    train_labels_svm
)

print("\nReconstruction complete.")
print("Classes:", svm_model.classes_)
print("Coefficient shape:", svm_model.coef_.shape)
print("Intercept shape:", svm_model.intercept_.shape)

In [ ]:
# Cell 59 — Final held-out test evaluation
# TF-IDF + Linear SVM

print("=" * 70)
print("FINAL TEST EVALUATION — TF-IDF + LINEAR SVM")
print("=" * 70)

# Transform using the frozen training-only TF-IDF vocabulary.
X_test_tfidf_svm = tfidf_vectorizer.transform(
    test_df["comment"].astype(str)
)

print("Test matrix shape:", X_test_tfidf_svm.shape)

# Predict using the reconstructed SVM.
svm_test_pred = svm_model.predict(
    X_test_tfidf_svm
)

svm_test_true = (
    test_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("\nClassification Report:")
print(
    classification_report(
        svm_test_true,
        svm_test_pred,
        target_names=["Negative", "Neutral", "Positive"],
        digits=4,
        zero_division=0
    )
)

svm_test_cm = confusion_matrix(
    svm_test_true,
    svm_test_pred
)

print("Confusion Matrix:")
print(svm_test_cm)

print("\nRows = Actual")
print("Columns = Predicted")

svm_test_accuracy = accuracy_score(
    svm_test_true,
    svm_test_pred
)

svm_test_precision, svm_test_recall, svm_test_f1, _ = (
    precision_recall_fscore_support(
        svm_test_true,
        svm_test_pred,
        average="macro",
        zero_division=0
    )
)

_, _, svm_test_f1_weighted, _ = (
    precision_recall_fscore_support(
        svm_test_true,
        svm_test_pred,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {svm_test_accuracy:.4f}")
print(f"Macro Precision:  {svm_test_precision:.4f}")
print(f"Macro Recall:     {svm_test_recall:.4f}")
print(f"Macro F1:         {svm_test_f1:.4f}")
print(f"Weighted F1:      {svm_test_f1_weighted:.4f}")

In [ ]:
# Cell 60 — Locate the trained CNN model

import tensorflow as tf

print("=" * 70)
print("LOCATING TRAINED CNN MODEL")
print("=" * 70)

cnn_candidates = []

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, tf.keras.Model):
            cnn_candidates.append(name)
    except Exception:
        pass

print("Keras model objects found:", cnn_candidates)

for name in cnn_candidates:
    obj = globals()[name]
    print(
        f"{name}: "
        f"parameters={obj.count_params():,}"
    )

In [ ]:
# Cell 61 — Check for saved CNN model artifacts

import os

print("=" * 70)
print("CHECKING FOR SAVED CNN ARTIFACTS")
print("=" * 70)

cnn_files = []

for base_path in ["/content"]:
    if os.path.exists(base_path):
        for root, dirs, files in os.walk(base_path):
            for filename in files:
                lower_name = filename.lower()

                if (
                    lower_name.endswith((".keras", ".h5", ".hdf5"))
                    or "cnn" in lower_name
                    or "tokenizer" in lower_name
                ):
                    cnn_files.append(
                        os.path.join(root, filename)
                    )

if cnn_files:
    print("Potential CNN/tokenizer artifacts found:")
    for path in sorted(cnn_files):
        print(" -", path)
else:
    print("No saved CNN artifacts found.")

In [ ]:
# Cell 62 — Check for retained CNN tokenizer/history objects

print("=" * 70)
print("CHECKING RETAINED CNN TRAINING STATE")
print("=" * 70)

global_items = list(globals().items())

tokenizer_candidates = []
history_candidates = []

for name, obj in global_items:

    # Keras/TensorFlow tokenizer-like objects
    if hasattr(obj, "word_index") and hasattr(obj, "texts_to_sequences"):
        tokenizer_candidates.append(name)

    # Keras History objects
    if hasattr(obj, "history") and isinstance(
        getattr(obj, "history", None), dict
    ):
        history_candidates.append(name)

print("Tokenizer-like objects found:", tokenizer_candidates)
print("History-like objects found:", history_candidates)

for name in tokenizer_candidates:
    obj = globals()[name]
    print(
        f"\nTokenizer: {name}"
    )
    print("Vocabulary size:", len(obj.word_index))

for name in history_candidates:
    obj = globals()[name]
    print(
        f"\nHistory: {name}"
    )
    print("History keys:", list(obj.history.keys()))

    if "val_accuracy" in obj.history:
        print(
            "Validation accuracies:",
            [
                round(float(x), 4)
                for x in obj.history["val_accuracy"]
            ]
        )

    if "val_loss" in obj.history:
        print(
            "Validation losses:",
            [
                round(float(x), 4)
                for x in obj.history["val_loss"]
            ]
        )

In [ ]:
# Cell 63 — Reconstruct CNN tokenizer from training data only

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 70)
print("RECONSTRUCTING CNN TOKENIZER")
print("=" * 70)

CNN_MAX_VOCAB = 50000
CNN_MAX_SEQUENCE_LENGTH = 150

cnn_tokenizer_reconstructed = Tokenizer(
    num_words=CNN_MAX_VOCAB,
    oov_token="<OOV>"
)

cnn_tokenizer_reconstructed.fit_on_texts(
    train_df["comment"].astype(str).tolist()
)

actual_vocab_size = min(
    CNN_MAX_VOCAB,
    len(cnn_tokenizer_reconstructed.word_index) + 1
)

print("Maximum vocabulary:", CNN_MAX_VOCAB)
print("Actual vocabulary size:", actual_vocab_size)
print("Maximum sequence length:", CNN_MAX_SEQUENCE_LENGTH)

# Convert frozen train/validation text to sequences
X_train_cnn_reconstructed = pad_sequences(
    cnn_tokenizer_reconstructed.texts_to_sequences(
        train_df["comment"].astype(str).tolist()
    ),
    maxlen=CNN_MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_cnn_reconstructed = pad_sequences(
    cnn_tokenizer_reconstructed.texts_to_sequences(
        val_df["comment"].astype(str).tolist()
    ),
    maxlen=CNN_MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train_cnn_reconstructed = (
    train_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

y_val_cnn_reconstructed = (
    val_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("\nTrain shape:", X_train_cnn_reconstructed.shape)
print("Validation shape:", X_val_cnn_reconstructed.shape)

print("\nLabels:")
print("Train:", np.bincount(y_train_cnn_reconstructed))
print("Validation:", np.bincount(y_val_cnn_reconstructed))

In [ ]:
# Cell 64 — Reconstruct the CNN architecture

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)

print("=" * 70)
print("RECONSTRUCTING CNN MODEL")
print("=" * 70)

cnn_reconstructed = Sequential([
    Input(
        shape=(CNN_MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),

    Embedding(
        input_dim=actual_vocab_size,
        output_dim=128
    ),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    GlobalMaxPooling1D(),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        3,
        activation="softmax"
    )
])

cnn_reconstructed.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Architecture:")
cnn_reconstructed.summary()

print(
    "\nTotal parameters:",
    f"{cnn_reconstructed.count_params():,}"
)

print("Optimizer: Adam")
print("Loss: sparse categorical crossentropy")
print("Classes: 3")

In [ ]:
# Cell 65 — Retrain reconstructed CNN
# Training and validation only; test set remains untouched.

from tensorflow.keras.callbacks import EarlyStopping

print("=" * 70)
print("TRAINING RECONSTRUCTED CNN")
print("=" * 70)

cnn_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

cnn_reconstructed_history = cnn_reconstructed.fit(
    X_train_cnn_reconstructed,
    y_train_cnn_reconstructed,

    validation_data=(
        X_val_cnn_reconstructed,
        y_val_cnn_reconstructed
    ),

    epochs=4,
    batch_size=32,
    callbacks=[cnn_early_stopping],
    verbose=1
)

print("\n" + "=" * 70)
print("RECONSTRUCTED CNN TRAINING COMPLETE")
print("=" * 70)

print(
    "Epochs actually completed:",
    len(cnn_reconstructed_history.history["loss"])
)

print(
    "Best validation accuracy:",
    round(
        max(cnn_reconstructed_history.history["val_accuracy"]),
        4
    )
)

print(
    "Best validation loss:",
    round(
        min(cnn_reconstructed_history.history["val_loss"]),
        4
    )
)

print("\nNOTE:")
print("This is a reconstructed CNN because the original CNN weights")
print("were not saved in the notebook or available as a checkpoint.")

In [ ]:
# Cell 66 — Prepare held-out test sequences for the reconstructed CNN

print("=" * 70)
print("PREPARING HELD-OUT TEST SET — RECONSTRUCTED CNN")
print("=" * 70)

X_test_cnn_reconstructed = pad_sequences(
    cnn_tokenizer_reconstructed.texts_to_sequences(
        test_df["comment"].astype(str).tolist()
    ),
    maxlen=CNN_MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_test_cnn_reconstructed = (
    test_df["sentiment"]
    .map({
        "Negative": 0,
        "Neutral": 1,
        "Positive": 2
    })
    .astype(int)
    .to_numpy()
)

print("Test shape:", X_test_cnn_reconstructed.shape)
print("Test labels:", np.bincount(y_test_cnn_reconstructed))
print("Test examples:", len(y_test_cnn_reconstructed))

In [ ]:
# Cell 67 — Final held-out test evaluation
# Reconstructed CNN

print("=" * 70)
print("FINAL TEST EVALUATION — RECONSTRUCTED CNN")
print("=" * 70)

# Predict probabilities
cnn_test_probabilities = cnn_reconstructed.predict(
    X_test_cnn_reconstructed,
    batch_size=256,
    verbose=1
)

# Convert probabilities to class predictions
cnn_test_pred = np.argmax(
    cnn_test_probabilities,
    axis=1
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_cnn_reconstructed,
        cnn_test_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ],
        digits=4,
        zero_division=0
    )
)

cnn_test_cm = confusion_matrix(
    y_test_cnn_reconstructed,
    cnn_test_pred
)

print("Confusion Matrix:")
print(cnn_test_cm)

print("\nRows = Actual")
print("Columns = Predicted")

cnn_test_accuracy = accuracy_score(
    y_test_cnn_reconstructed,
    cnn_test_pred
)

cnn_test_precision, cnn_test_recall, cnn_test_f1, _ = (
    precision_recall_fscore_support(
        y_test_cnn_reconstructed,
        cnn_test_pred,
        average="macro",
        zero_division=0
    )
)

_, _, cnn_test_f1_weighted, _ = (
    precision_recall_fscore_support(
        y_test_cnn_reconstructed,
        cnn_test_pred,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {cnn_test_accuracy:.4f}")
print(f"Macro Precision:  {cnn_test_precision:.4f}")
print(f"Macro Recall:     {cnn_test_recall:.4f}")
print(f"Macro F1:         {cnn_test_f1:.4f}")
print(f"Weighted F1:      {cnn_test_f1_weighted:.4f}")

In [ ]:
# Cell 68 — Locate the trained BERT baseline model

from transformers import BertForSequenceClassification

print("=" * 70)
print("LOCATING TRAINED BERT BASELINE")
print("=" * 70)

bert_candidates = []

for name, obj in list(globals().items()):
    try:
        is_bert = isinstance(obj, BertForSequenceClassification)
    except Exception:
        is_bert = False

    if is_bert:
        bert_candidates.append(name)

print("BERT sequence-classification models found:", bert_candidates)

for name in bert_candidates:
    obj = globals()[name]
    total_params = sum(
        p.numel() for p in obj.parameters()
    )

    print(
        f"{name}: "
        f"parameters={total_params:,}, "
        f"device={next(obj.parameters()).device}"
    )

In [ ]:
# Cell 69 — Prepare held-out test dataset for the plain BERT baseline

print("=" * 70)
print("PREPARING HELD-OUT TEST SET — BERT BASELINE")
print("=" * 70)

# Start from the already tokenized test dataset.
# Remove the topic feature because plain BERT does not use it.
bert_test_dataset = test_bert_dataset.remove_columns(
    ["topic_id"]
)

print("Test examples:", len(bert_test_dataset))
print("Features:")
print(bert_test_dataset.features)

print("\nTest labels preserved:", "label" in bert_test_dataset.column_names)
print("Topic feature used by BERT:", "topic_id" in bert_test_dataset.column_names)

In [ ]:
# Cell 70 — Final held-out test evaluation
# BERT baseline

print("=" * 70)
print("FINAL TEST EVALUATION — BERT BASELINE")
print("=" * 70)

# Use the original BERT Trainer/model.
# No retraining or parameter updates occur during prediction.
bert_test_output = bert_trainer.predict(
    bert_test_dataset
)

bert_test_logits = bert_test_output.predictions

if isinstance(bert_test_logits, tuple):
    bert_test_logits = bert_test_logits[0]

bert_test_pred = np.argmax(
    bert_test_logits,
    axis=-1
)

bert_test_true = np.array(
    bert_test_dataset["label"]
)

label_names = [
    "Negative",
    "Neutral",
    "Positive"
]

print("\nClassification Report:")
print(
    classification_report(
        bert_test_true,
        bert_test_pred,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

bert_test_cm = confusion_matrix(
    bert_test_true,
    bert_test_pred
)

print("Confusion Matrix:")
print(bert_test_cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("Label order:", label_names)

bert_test_accuracy = accuracy_score(
    bert_test_true,
    bert_test_pred
)

bert_test_precision, bert_test_recall, bert_test_f1, _ = (
    precision_recall_fscore_support(
        bert_test_true,
        bert_test_pred,
        average="macro",
        zero_division=0
    )
)

_, _, bert_test_f1_weighted, _ = (
    precision_recall_fscore_support(
        bert_test_true,
        bert_test_pred,
        average="weighted",
        zero_division=0
    )
)

print("\nOverall Test Metrics:")
print(f"Accuracy:         {bert_test_accuracy:.4f}")
print(f"Macro Precision:  {bert_test_precision:.4f}")
print(f"Macro Recall:     {bert_test_recall:.4f}")
print(f"Macro F1:         {bert_test_f1:.4f}")
print(f"Weighted F1:      {bert_test_f1_weighted:.4f}")

In [ ]:
# Cell 71: Final publication-ready held-out test results

final_results = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": 0.8196,
        "Macro Precision": 0.8209,
        "Macro Recall": 0.8206,
        "Macro F1": 0.8203,
        "Weighted F1": 0.8196,
        "Status": "Reconstructed from frozen split"
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Accuracy": 0.8376,
        "Macro Precision": 0.8382,
        "Macro Recall": 0.8383,
        "Macro F1": 0.8382,
        "Weighted F1": 0.8375,
        "Status": "Reconstructed from frozen split"
    },
    {
        "Model": "CNN",
        "Accuracy": 0.8735,
        "Macro Precision": 0.8761,
        "Macro Recall": 0.8748,
        "Macro F1": 0.8754,
        "Weighted F1": 0.8736,
        "Status": "Reconstructed model"
    },
    {
        "Model": "BERT",
        "Accuracy": 0.9268,
        "Macro Precision": 0.9293,
        "Macro Recall": 0.9269,
        "Macro F1": 0.9277,
        "Weighted F1": 0.9270,
        "Status": "Trained model"
    },
    {
        "Model": "BERT-CNN + Topic",
        "Accuracy": 0.9229,
        "Macro Precision": 0.9265,
        "Macro Recall": 0.9229,
        "Macro F1": 0.9238,
        "Weighted F1": 0.9231,
        "Status": "Trained model"
    }
])

# Display as percentages
display(
    final_results.style
    .format({
        "Accuracy": "{:.2%}",
        "Macro Precision": "{:.2%}",
        "Macro Recall": "{:.2%}",
        "Macro F1": "{:.2%}",
        "Weighted F1": "{:.2%}"
    })
)

# Save the structured results
FINAL_RESULTS_PATH = "/content/final_five_model_test_results.csv"
final_results.to_csv(FINAL_RESULTS_PATH, index=False)

print(f"\nSaved: {FINAL_RESULTS_PATH}")

In [ ]:
# Cell 72: Per-class test F1 comparison

per_class_f1 = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM",
        "CNN",
        "BERT",
        "BERT-CNN + Topic"
    ],
    "Negative F1": [0.8108, 0.8310, 0.8572, 0.9209, 0.9174],
    "Neutral F1":  [0.8338, 0.8517, 0.9120, 0.9429, 0.9387],
    "Positive F1": [0.8162, 0.8320, 0.8571, 0.9193, 0.9154]
})

display(
    per_class_f1.style.format({
        "Negative F1": "{:.2%}",
        "Neutral F1": "{:.2%}",
        "Positive F1": "{:.2%}"
    })
)

# Save for later publication/reporting
PER_CLASS_PATH = "/content/final_per_class_f1_results.csv"
per_class_f1.to_csv(PER_CLASS_PATH, index=False)

print(f"\nSaved: {PER_CLASS_PATH}")

In [ ]:
# Cell 73: Final held-out test confusion matrices

confusion_matrices = {
    "TF-IDF + Logistic Regression": np.array([
        [2856, 269, 324],
        [293, 2475, 163],
        [447, 262, 2656]
    ]),

    "TF-IDF + Linear SVM": np.array([
        [2883, 226, 340],
        [236, 2503, 192],
        [371, 218, 2776]
    ]),

    "CNN": np.array([
        [2984, 109, 356],
        [163, 2647, 121],
        [366, 118, 2881]
    ]),

    "BERT": np.array([
        [3144, 50, 255],
        [105, 2709, 117],
        [130, 56, 3179]
    ]),

    "BERT-CNN + Topic": np.array([
        [3094, 55, 300],
        [102, 2686, 143],
        [100, 51, 3214]
    ])
}

# Verify matrix dimensions and totals
expected_test_size = len(test_df)

for model_name, cm in confusion_matrices.items():
    assert cm.shape == (3, 3), f"{model_name}: unexpected matrix shape"
    assert cm.sum() == expected_test_size, (
        f"{model_name}: matrix total {cm.sum()} != test size {expected_test_size}"
    )

print(f"Test set size: {expected_test_size:,}")
print("All five confusion matrices verified successfully.\n")

for model_name, cm in confusion_matrices.items():
    print(model_name)
    print("Rows = Actual | Columns = Predicted")
    print("             Negative  Neutral  Positive")
    print(f"Negative     {cm[0,0]:8d} {cm[0,1]:8d} {cm[0,2]:8d}")
    print(f"Neutral      {cm[1,0]:8d} {cm[1,1]:8d} {cm[1,2]:8d}")
    print(f"Positive     {cm[2,0]:8d} {cm[2,1]:8d} {cm[2,2]:8d}")
    print()

In [ ]:
# Cell 74: Publication-ready model comparison chart

import matplotlib.pyplot as plt
import numpy as np

models = final_results["Model"].tolist()
accuracy = final_results["Accuracy"].values * 100
macro_f1 = final_results["Macro F1"].values * 100

x = np.arange(len(models))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(
    x - width/2,
    accuracy,
    width,
    label="Accuracy"
)

bars2 = ax.bar(
    x + width/2,
    macro_f1,
    width,
    label="Macro F1"
)

ax.set_ylabel("Score (%)")
ax.set_title("Held-Out Test Performance of Five Sentiment Models")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha="right")
ax.set_ylim(75, 100)
ax.legend()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.2f}%",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

CHART_PATH = "/content/five_model_test_comparison.png"
plt.savefig(CHART_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {CHART_PATH}")

In [ ]:
# Cell 75: Validation vs. held-out test performance

validation_test_summary = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Validation Accuracy": 0.8139,
        "Validation Macro F1": 0.8140,
        "Test Accuracy": 0.8196,
        "Test Macro F1": 0.8203
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Validation Accuracy": 0.8390,
        "Validation Macro F1": 0.8393,
        "Test Accuracy": 0.8376,
        "Test Macro F1": 0.8382
    },
    {
        "Model": "CNN",
        "Validation Accuracy": 0.8747,
        "Validation Macro F1": 0.8764,
        "Test Accuracy": 0.8735,
        "Test Macro F1": 0.8754
    },
    {
        "Model": "BERT",
        "Validation Accuracy": 0.9295,
        "Validation Macro F1": 0.9298,
        "Test Accuracy": 0.9268,
        "Test Macro F1": 0.9277
    },
    {
        "Model": "BERT-CNN + Topic",
        "Validation Accuracy": 0.9201,
        "Validation Macro F1": 0.9208,
        "Test Accuracy": 0.9229,
        "Test Macro F1": 0.9238
    }
])

display(
    validation_test_summary.style.format({
        "Validation Accuracy": "{:.2%}",
        "Validation Macro F1": "{:.2%}",
        "Test Accuracy": "{:.2%}",
        "Test Macro F1": "{:.2%}"
    })
)

# Save artifact
VALIDATION_TEST_PATH = "/content/validation_vs_test_summary.csv"
validation_test_summary.to_csv(VALIDATION_TEST_PATH, index=False)

print(f"\nSaved: {VALIDATION_TEST_PATH}")

In [ ]:
# Cell 76: Validation-to-test performance change

validation_test_delta = validation_test_summary.copy()

validation_test_delta["Accuracy Change (pp)"] = (
    validation_test_delta["Test Accuracy"]
    - validation_test_delta["Validation Accuracy"]
) * 100

validation_test_delta["Macro F1 Change (pp)"] = (
    validation_test_delta["Test Macro F1"]
    - validation_test_delta["Validation Macro F1"]
) * 100

delta_display = validation_test_delta[
    [
        "Model",
        "Validation Accuracy",
        "Test Accuracy",
        "Accuracy Change (pp)",
        "Validation Macro F1",
        "Test Macro F1",
        "Macro F1 Change (pp)"
    ]
]

display(
    delta_display.style.format({
        "Validation Accuracy": "{:.2%}",
        "Test Accuracy": "{:.2%}",
        "Accuracy Change (pp)": "{:+.2f}",
        "Validation Macro F1": "{:.2%}",
        "Test Macro F1": "{:.2%}",
        "Macro F1 Change (pp)": "{:+.2f}"
    })
)

# Save artifact
DELTA_PATH = "/content/validation_test_performance_delta.csv"
validation_test_delta.to_csv(DELTA_PATH, index=False)

print(f"\nSaved: {DELTA_PATH}")

In [ ]:
# Cell 77: Final consolidated research results

final_research_results = final_results.merge(
    validation_test_summary[
        [
            "Model",
            "Validation Accuracy",
            "Validation Macro F1"
        ]
    ],
    on="Model",
    how="left"
)

# Put columns in publication-friendly order
final_research_results = final_research_results[
    [
        "Model",
        "Validation Accuracy",
        "Accuracy",
        "Validation Macro F1",
        "Macro F1",
        "Macro Precision",
        "Macro Recall",
        "Weighted F1",
        "Status"
    ]
]

display(
    final_research_results.style.format({
        "Validation Accuracy": "{:.2%}",
        "Accuracy": "{:.2%}",
        "Validation Macro F1": "{:.2%}",
        "Macro F1": "{:.2%}",
        "Macro Precision": "{:.2%}",
        "Macro Recall": "{:.2%}",
        "Weighted F1": "{:.2%}"
    })
)

FINAL_RESEARCH_PATH = "/content/final_research_results.csv"
final_research_results.to_csv(FINAL_RESEARCH_PATH, index=False)

print(f"\nSaved: {FINAL_RESEARCH_PATH}")
print("\nFinal test set remains untouched: 9,745 observations.")

In [ ]:
# Cell 77 — Recreate final five-model test results
# No model training required.

import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM",
        "CNN",
        "BERT",
        "BERT-CNN + Topic"
    ],
    "Accuracy": [0.8196, 0.8376, 0.8735, 0.9268, 0.9229],
    "Macro Precision": [0.8209, 0.8382, 0.8761, 0.9293, 0.9265],
    "Macro Recall": [0.8206, 0.8383, 0.8748, 0.9269, 0.9229],
    "Macro F1": [0.8203, 0.8382, 0.8754, 0.9277, 0.9238],
    "Weighted F1": [0.8196, 0.8375, 0.8736, 0.9270, 0.9231]
})

display(
    final_results.style.format({
        "Accuracy": "{:.2%}",
        "Macro Precision": "{:.2%}",
        "Macro Recall": "{:.2%}",
        "Macro F1": "{:.2%}",
        "Weighted F1": "{:.2%}"
    })
)

In [ ]:
# Cell 78 — Save final test results

FINAL_RESULTS_PATH = "/content/final_five_model_test_results.csv"

final_results.to_csv(FINAL_RESULTS_PATH, index=False)

print(f"Saved successfully: {FINAL_RESULTS_PATH}")

In [ ]:
# Cell 79 — Final test-set comparison plot

import matplotlib.pyplot as plt
import numpy as np

models = final_results["Model"]
accuracy = final_results["Accuracy"] * 100
macro_f1 = final_results["Macro F1"] * 100

x = np.arange(len(models))
width = 0.36

plt.figure(figsize=(12, 6))

plt.bar(x - width/2, accuracy, width, label="Accuracy")
plt.bar(x + width/2, macro_f1, width, label="Macro F1")

plt.xticks(x, models, rotation=20, ha="right")
plt.ylabel("Score (%)")
plt.title("Five-Model Comparison on the Frozen Reddit Test Set")
plt.ylim(75, 100)
plt.legend()
plt.tight_layout()

FIGURE_PATH = "/content/five_model_test_comparison.png"
plt.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved successfully: {FIGURE_PATH}")

In [ ]:
# Cell 80 — Per-class F1 scores on the frozen Reddit test set

per_class_f1 = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM",
        "CNN",
        "BERT",
        "BERT-CNN + Topic"
    ],
    "Negative F1": [0.8108, 0.8310, 0.8572, 0.9209, 0.9174],
    "Neutral F1": [0.8338, 0.8517, 0.9120, 0.9429, 0.9387],
    "Positive F1": [0.8162, 0.8320, 0.8571, 0.9193, 0.9154]
})

display(
    per_class_f1.style.format({
        "Negative F1": "{:.2%}",
        "Neutral F1": "{:.2%}",
        "Positive F1": "{:.2%}"
    })
)

In [ ]:
# Cell 81 — Save per-class F1 results

PER_CLASS_PATH = "/content/final_per_class_f1_results.csv"

per_class_f1.to_csv(PER_CLASS_PATH, index=False)

print(f"Saved successfully: {PER_CLASS_PATH}")

In [ ]:
# Cell 82 — Test-set confusion matrices

confusion_matrices = {
    "TF-IDF + Logistic Regression": [
        [2856, 269, 324],
        [293, 2475, 163],
        [447, 262, 2656]
    ],

    "TF-IDF + Linear SVM": [
        [2883, 226, 340],
        [236, 2503, 192],
        [371, 218, 2776]
    ],

    "CNN": [
        [2984, 109, 356],
        [163, 2647, 121],
        [366, 118, 2881]
    ],

    "BERT": [
        [3144, 50, 255],
        [105, 2709, 117],
        [130, 56, 3179]
    ],

    "BERT-CNN + Topic": [
        [3094, 55, 300],
        [102, 2686, 143],
        [100, 51, 3214]
    ]
}

print("Five confusion matrices loaded successfully.")
print("\nClass order: Negative, Neutral, Positive")

for model, cm in confusion_matrices.items():
    print(f"\n{model}")
    display(pd.DataFrame(
        cm,
        index=["Actual Negative", "Actual Neutral", "Actual Positive"],
        columns=["Pred Negative", "Pred Neutral", "Pred Positive"]
    ))

In [ ]:
# Cell 83 — Five-model confusion matrix figure

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

class_names = ["Negative", "Neutral", "Positive"]

for i, (model, cm) in enumerate(confusion_matrices.items()):
    cm = np.array(cm)
    ax = axes[i]

    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(model, fontsize=12, fontweight="bold")

    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(3))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    # Add cell values
    for row in range(3):
        for col in range(3):
            ax.text(
                col,
                row,
                f"{cm[row, col]:,}",
                ha="center",
                va="center",
                fontsize=11
            )

# Remove unused sixth subplot
fig.delaxes(axes[5])

fig.suptitle(
    "Confusion Matrices of Five Sentiment Classification Models",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

CM_FIGURE_PATH = "/content/five_model_confusion_matrices.png"
plt.savefig(CM_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved successfully: {CM_FIGURE_PATH}")


In [ ]:
# Cell 84 — Validation vs. test performance

validation_test = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM",
        "CNN",
        "BERT",
        "BERT-CNN + Topic"
    ],
    "Validation Accuracy": [
        0.8139, 0.8390, 0.8747, 0.9295, 0.9201
    ],
    "Validation Macro F1": [
        0.8140, 0.8393, 0.8764, 0.9298, 0.9208
    ],
    "Test Accuracy": [
        0.8196, 0.8376, 0.8735, 0.9268, 0.9229
    ],
    "Test Macro F1": [
        0.8203, 0.8382, 0.8754, 0.9277, 0.9238
    ]
})

display(
    validation_test.style.format({
        "Validation Accuracy": "{:.2%}",
        "Validation Macro F1": "{:.2%}",
        "Test Accuracy": "{:.2%}",
        "Test Macro F1": "{:.2%}"
    })
)

In [ ]:
# Cell 85 — Calculate validation-to-test differences

validation_test_delta = validation_test.copy()

validation_test_delta["Accuracy Delta (pp)"] = (
    validation_test_delta["Test Accuracy"]
    - validation_test_delta["Validation Accuracy"]
) * 100

validation_test_delta["Macro F1 Delta (pp)"] = (
    validation_test_delta["Test Macro F1"]
    - validation_test_delta["Validation Macro F1"]
) * 100

display(
    validation_test_delta[
        ["Model", "Accuracy Delta (pp)", "Macro F1 Delta (pp)"]
    ].style.format({
        "Accuracy Delta (pp)": "{:+.2f}",
        "Macro F1 Delta (pp)": "{:+.2f}"
    })
)

In [ ]:
# Cell 86 — Save validation/test comparison results

VALIDATION_TEST_PATH = "/content/validation_vs_test_summary.csv"
DELTA_PATH = "/content/validation_test_performance_delta.csv"

validation_test.to_csv(VALIDATION_TEST_PATH, index=False)
validation_test_delta.to_csv(DELTA_PATH, index=False)

print("Saved successfully:")
print(VALIDATION_TEST_PATH)
print(DELTA_PATH)

In [ ]:
# Cell 87 — Final research results table

final_research_results = final_results.copy()

# Add validation metrics
validation_lookup = validation_test[
    ["Model", "Validation Accuracy", "Validation Macro F1"]
]

final_research_results = final_research_results.merge(
    validation_lookup,
    on="Model",
    how="left"
)

# Reorder columns
final_research_results = final_research_results[
    [
        "Model",
        "Validation Accuracy",
        "Validation Macro F1",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
]

display(
    final_research_results.style.format({
        "Validation Accuracy": "{:.2%}",
        "Validation Macro F1": "{:.2%}",
        "Accuracy": "{:.2%}",
        "Macro Precision": "{:.2%}",
        "Macro Recall": "{:.2%}",
        "Macro F1": "{:.2%}",
        "Weighted F1": "{:.2%}"
    })
)

In [ ]:
# Cell 88 — Save final consolidated research results

FINAL_RESEARCH_PATH = "/content/final_research_results.csv"

final_research_results.to_csv(
    FINAL_RESEARCH_PATH,
    index=False
)

print(f"Saved successfully: {FINAL_RESEARCH_PATH}")

In [ ]:
# Cell 89 — Experiment metadata / reproducibility record

experiment_metadata = {
    "dataset": "Reddit Russia-Ukraine conflict dataset",
    "usable_comments": 63994,
    "train_size": 44085,
    "validation_size": 10164,
    "test_size": 9745,
    "split_method": "GroupShuffleSplit by Reddit post/thread ID",
    "random_state": 42,
    "sentiment_labels": "VADER: Negative / Neutral / Positive",
    "vader_thresholds": "compound >= 0.05 Positive; <= -0.05 Negative; otherwise Neutral",
    "manual_vader_validation_sample": 100,
    "manual_validation_accuracy": 0.8681,
    "manual_validation_cohen_kappa": 0.8022,
    "topic_count": 10,
    "topic_embedding_model": "all-MiniLM-L6-v2",
    "topic_clustering": "KMeans",
    "topic_random_state": 42,
    "bert_model": "bert-base-uncased",
    "bert_max_length": 128,
    "cnn_max_sequence_length": 150,
    "cnn_vocab_limit": 50000,
    "proposed_model": "BERT-CNN + Topic",
    "test_set_used_for_model_selection": False,
    "cnn_result_status": "Reconstructed from recorded architecture and settings; original weights unavailable"
}

experiment_metadata_df = pd.DataFrame(
    list(experiment_metadata.items()),
    columns=["Parameter", "Value"]
)

display(experiment_metadata_df)

In [ ]:
# Cell 90 — Save experiment metadata

METADATA_PATH = "/content/experiment_metadata.csv"

experiment_metadata_df.to_csv(
    METADATA_PATH,
    index=False
)

print(f"Saved successfully: {METADATA_PATH}")